# [문제 4-2] 조건부 분기가 있는 메뉴 추천 시스템 (LangGraph 사용)

이 노트북은 LangGraph를 사용하여 고객의 질문 의도를 파악하고, 의도에 따라 다른 로직으로 분기하여 답변하는 카페 메뉴 추천 시스템을 구현합니다.

**학습 목표:**
- `MessagesState`를 사용하여 대화 상태 관리하기
- 키워드 기반으로 사용자 질문의 의도를 분류하는 라우팅(Routing) 로직 구현하기
- LangGraph의 `add_conditional_edges`를 사용하여 조건부 분기 그래프 생성하기
- 의도별로 다른 검색 전략(의미론적 검색, 키워드 검색, 폴백 검색)을 적용하기
- 정규표현식을 사용하여 검색된 문서에서 구조화된 정보 추출하기

In [1]:
# (1) API 키 및 기본 라이브러리 설정
import os
import re
from typing import List, Literal, Tuple
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import MessagesState, StateGraph, START, END

# .env 파일에서 환경 변수 로드
load_dotenv()

# API 키 확인
if "UPSTAGE_API_KEY" not in os.environ:
    os.environ["UPSTAGE_API_KEY"] = input("Enter your Upstage API key: ")

print("✅ API 키가 성공적으로 로드되었습니다.")

✅ API 키가 성공적으로 로드되었습니다.


### (2) 벡터 DB 로드

이전 문제(4-1)에서 `cafe_menu.txt` 파일로 생성했던 **Chroma** 벡터 데이터베이스를 재사용합니다. `persist_directory`를 지정하여 저장된 인덱스를 불러옵니다.

In [2]:
from langchain_community.vectorstores import Chroma
from langchain_upstage import UpstageEmbeddings

# 임베딩 모델 초기화
embeddings_model = UpstageEmbeddings(model="solar-embedding-1-large")
db_dir = "../db/cafe_db"

# 이전에 저장된 Chroma DB 로드
try:
    cafe_db = Chroma(
        embedding_function=embeddings_model,
        persist_directory=db_dir
    )
    print(f"✅ Chroma 벡터 DB를 성공적으로 로드했습니다. (경로: {db_dir})")
    print(f" - DB에 저장된 메뉴 수: {cafe_db._collection.count()}개")
except Exception as e:
    print(f"🚨 DB 로드 중 오류 발생: {e}")

/var/folders/_b/pqs_x7fd3pl16l4x573_l2th0000gn/T/ipykernel_24658/334098039.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  cafe_db = Chroma(


✅ Chroma 벡터 DB를 성공적으로 로드했습니다. (경로: ../db/cafe_db)
 - DB에 저장된 메뉴 수: 10개


### (1) 문의 유형 분류 및 정보 추출 함수

- **`classify_query`**: 키워드를 기반으로 사용자의 질문을 '메뉴 문의', '가격 문의', '추천 요청', '기타' 중 하나로 분류합니다. 이 함수의 반환 값에 따라 그래프가 분기됩니다.
- **`extract_menu_info`**: Chroma DB에서 검색된 `Document` 객체로부터 정규표현식을 사용해 메뉴 이름, 가격, 설명을 추출하여 구조화된 데이터로 만듭니다.

### (2) 문의 유형별 응답 생성 함수

각 문의 유형에 따라 다른 검색 전략을 사용하여 응답을 생성하는 함수들을 정의합니다. 이 함수들은 LangGraph에서 각각의 **노드(Node)** 로 작동하게 됩니다.

### (1) 상태 및 그래프 정의

`MessagesState`를 상태로 사용하고, 첫 번째 노드로 질문을 분류하는 `classify_node`를 설정합니다. 그 후 `add_conditional_edges`를 사용해 분류 결과에 따라 서로 다른 응답 생성 노드로 연결합니다.

In [3]:
from langgraph.graph import StateGraph, START, END
from typing import List
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import AnyMessage, AIMessage
import operator

# --- 1. 모든 핵심 함수 및 상태 정의 (통합) ---

# 1-1. 상태(State) 정의
class GraphState(TypedDict):
    messages: Annotated[List[AnyMessage], operator.add]
    classification_result: str

# 1-2. 문의 유형 분류 및 정보 추출 함수
def classify_query(query: str) -> Literal["메뉴 문의", "가격 문의", "추천 요청", "기타"]:
    query_lower = query.lower()
    if any(keyword in query_lower for keyword in ["가격", "얼마"]):
        return "가격 문의"
    elif any(keyword in query_lower for keyword in ["추천", "어때", "골라줘"]):
        return "추천 요청"
    elif any(keyword in query_lower for keyword in ["재료", "설명", "뭐야", "뭐가 들어있어"]):
        return "메뉴 문의"
    else:
        return "기타"

def extract_menu_info(doc: Document) -> dict:
    content = doc.page_content
    menu_name = doc.metadata.get('menu_name', 'Unknown')
    price_match = re.search(r'가격:\s*(₩[\d,]+)', content)
    description_match = re.search(r'설명:\s*(.+)', content, re.DOTALL)
    return {
        "name": menu_name,
        "price": price_match.group(1) if price_match else "가격 정보 없음",
        "description": description_match.group(1).strip() if description_match else "설명 없음"
    }

# 1-3. 문의 유형별 응답 생성 함수 (state 타입을 GraphState로 수정)
def handle_price_query(state: GraphState) -> dict:
    docs = cafe_db.similarity_search("메뉴 가격", k=5)
    response_parts = ["카페의 주요 메뉴 가격 정보입니다:\n"]
    for doc in docs:
        info = extract_menu_info(doc)
        response_parts.append(f"- {info['name']}: {info['price']}")
    return {"messages": [AIMessage(content="\n".join(response_parts))]}

def handle_recommendation_request(state: GraphState) -> dict:
    user_message = state["messages"][-1].content
    docs = cafe_db.similarity_search(user_message, k=3)
    if not docs:
        docs = cafe_db.similarity_search("인기 메뉴", k=3)
    response_parts = ["이런 메뉴는 어떠신가요? 고객님들이 많이 찾으시는 메뉴입니다.\n"]
    for doc in docs:
        info = extract_menu_info(doc)
        response_parts.append(f"### {info['name']} ({info['price']})\n- {info['description']}")
    return {"messages": [AIMessage(content="\n".join(response_parts))]}

def handle_menu_query(state: GraphState) -> dict:
    user_message = state["messages"][-1].content
    docs = cafe_db.similarity_search(user_message, k=1)
    if docs:
        info = extract_menu_info(docs[0])
        response = (
            f"문의하신 '{info['name']}' 메뉴의 정보입니다.\n\n"
            f"- **가격**: {info['price']}\n"
            f"- **설명**: {info['description']}"
        )
    else:
        response = "죄송합니다, 문의하신 메뉴 정보를 찾을 수 없습니다."
    return {"messages": [AIMessage(content=response)]}

def handle_default(state: GraphState) -> dict:
    response = "죄송합니다. 저는 카페 메뉴에 대한 정보만 드릴 수 있어요. 메뉴, 가격, 추천에 대해 질문해주세요."
    return {"messages": [AIMessage(content=response)]}

# --- 2. 그래프 구성 ---

# 2-1. 분류 노드 정의
def classify_node(state: GraphState):
    last_message = state["messages"][-1].content
    classification = classify_query(last_message)
    print(f"🔍 분류 결과: {classification}")
    return {"classification_result": classification}

# 2-2. 그래프 빌더 생성 및 노드/엣지 설정
builder = StateGraph(GraphState)

builder.add_node("classify", classify_node)
builder.add_node("handle_menu_query", handle_menu_query)
builder.add_node("handle_price_query", handle_price_query)
builder.add_node("handle_recommendation", handle_recommendation_request)
builder.add_node("handle_default", handle_default)

builder.set_entry_point("classify")

builder.add_conditional_edges(
    "classify",
    lambda state: state["classification_result"],
    {
        "메뉴 문의": "handle_menu_query",
        "가격 문의": "handle_price_query",
        "추천 요청": "handle_recommendation",
        "기타": "handle_default",
    },
)

builder.add_edge("handle_menu_query", END)
builder.add_edge("handle_price_query", END)
builder.add_edge("handle_recommendation", END)
builder.add_edge("handle_default", END)

# --- 3. 그래프 컴파일 ---
graph = builder.compile()

print("✅ 조건부 분기 그래프 구성 완료!")

✅ 조건부 분기 그래프 구성 완료!


### (2) 그래프 시각화

생성된 그래프의 구조를 Mermaid 다이어그램으로 시각화하여 각 노드와 엣지의 흐름을 확인합니다.

In [4]:
def run_test(query):
    print(f"\n--- 💬 질문: {query} ---")
    inputs = {"messages": [HumanMessage(content=query)]}
    result = graph.invoke(inputs)
    final_response = result["messages"][-1]
    print(f"🤖 답변:\n{final_response.content}")

# 1. 메뉴 문의 테스트 (요구사항 질문)
run_test("아메리카노의 가격과 특징은 무엇인가요?")

# 2. 가격 문의 테스트
run_test("가장 비싼 메뉴는 얼마인가요?")

# 3. 추천 요청 테스트
run_test("달콤한 디저트 종류 추천해줘.")

# 4. 기타(분류 실패) 테스트
run_test("오늘 날씨 어때?")


--- 💬 질문: 아메리카노의 가격과 특징은 무엇인가요? ---
🔍 분류 결과: 가격 문의
🤖 답변:
카페의 주요 메뉴 가격 정보입니다:

- 티라미수: ₩7,500
- 바닐라 라떼: ₩6,000
- 프라푸치노: ₩7,000
- 카푸치노: ₩5,000
- 아이스 아메리카노: ₩4,500

--- 💬 질문: 가장 비싼 메뉴는 얼마인가요? ---
🔍 분류 결과: 가격 문의
🤖 답변:
카페의 주요 메뉴 가격 정보입니다:

- 티라미수: ₩7,500
- 바닐라 라떼: ₩6,000
- 프라푸치노: ₩7,000
- 카푸치노: ₩5,000
- 아이스 아메리카노: ₩4,500

--- 💬 질문: 달콤한 디저트 종류 추천해줘. ---
🔍 분류 결과: 추천 요청
🤖 답변:
이런 메뉴는 어떠신가요? 고객님들이 많이 찾으시는 메뉴입니다.

### 티라미수 (₩7,500)
- 이탈리아 전통 디저트로 마스카포네 치즈와 에스프레소에 적신 레이디핑거를 층층이 쌓아 만들었습니다. 부드럽고 달콤한 맛이 특징이며, 코코아 파우더로 마무리하여 깊은 풍미를 자랑합니다.
### 카라멜 마키아토 (₩6,500)
- 스팀 밀크 위에 에스프레소를 부어 만든 후 카라멜 시럽과 휘핑크림으로 마무리한 달콤한 커피입니다. 카라멜의 진한 단맛과 커피의 깊은 맛이 조화를 이루며, 시각적으로도 아름다운 층을 형성합니다.
### 바닐라 라떼 (₩6,000)
- 카페라떼에 달콤한 바닐라 시럽을 더한 인기 메뉴입니다. 바닐라의 달콤함과 커피의 쌉싸름함이 조화롭게 어우러지며, 휘핑크림 토핑으로 더욱 풍성한 맛을 즐길 수 있습니다.

--- 💬 질문: 오늘 날씨 어때? ---
🔍 분류 결과: 추천 요청
🤖 답변:
이런 메뉴는 어떠신가요? 고객님들이 많이 찾으시는 메뉴입니다.

### 아이스 아메리카노 (₩4,500)
- 진한 에스프레소에 차가운 물과 얼음을 넣어 만든 시원한 아이스 커피입니다. 깔끔하고 시원한 맛이 특징이며, 원두 본연의 풍미를 느낄 수 있습니다. 더운 날씨에 인기가 높습니다.
### 콜드브루 (₩5,000